In [2]:
!pip -q install chess

In [3]:
import chess
import chess.pgn
import chess.engine
import io
import json
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass, asdict
from enum import Enum

class MoveQuality(Enum):
    """Classification of move quality based on evaluation difference"""
    BEST = "best"           # Within 0.1 pawns of best move
    GOOD = "good"           # Within 0.3 pawns
    INACCURACY = "inaccuracy"  # 0.3-0.7 pawns worse
    MISTAKE = "mistake"     # 0.7-2.0 pawns worse
    BLUNDER = "blunder"     # More than 2.0 pawns worse

@dataclass
class MoveAnalysis:
    """Data structure for storing move analysis"""
    move_number: int
    move_played: str
    best_move: str
    eval_before: float
    eval_after: float
    eval_loss: float
    quality: MoveQuality
    best_line: str
    position_fen: str
    # New fields for comprehensive logging
    actual_move_line: str  # Line after actual move was played
    eval_after_best: float  # Evaluation after best move

@dataclass
class DetailedMoveLog:
    """Detailed logging structure for each move"""
    move_number: int
    position_fen: str
    actual_move_played: str
    best_move: str
    best_move_line: str
    eval_after_actual_move: float
    eval_after_best_move: float
    actual_move_inferior_line: str
    eval_difference: float
    is_player_move: bool
    move_quality: str

class ChessAnalyzer:
    def __init__(self, stockfish_path: str, depth: int = 15, enable_detailed_logging: bool = True):
        """
        Initialize the chess analyzer with Stockfish engine
        
        Args:
            stockfish_path: Path to Stockfish executable
            depth: Analysis depth for each position
            enable_detailed_logging: Whether to create detailed logs
        """
        self.stockfish_path = stockfish_path
        self.depth = depth
        self.engine = None
        self.board = chess.Board()
        self.enable_detailed_logging = enable_detailed_logging
        self.detailed_logs = []  # Store all detailed logs
        
    def __enter__(self):
        """Context manager entry - start engine"""
        self.engine = chess.engine.SimpleEngine.popen_uci(self.stockfish_path)
        return self
        
    def __exit__(self, exc_type, exc_val, exc_tb):
        """Context manager exit - close engine"""
        if self.engine:
            self.engine.quit()
            
    def classify_move_quality(self, eval_loss: float) -> MoveQuality:
        """
        Classify move quality based on evaluation loss
        
        Args:
            eval_loss: Centipawn loss (in pawns)
            
        Returns:
            MoveQuality enum
        """
        eval_loss = abs(eval_loss)
        if eval_loss <= 0.1:
            return MoveQuality.BEST
        elif eval_loss <= 0.3:
            return MoveQuality.GOOD
        elif eval_loss <= 0.5:
            return MoveQuality.INACCURACY
        elif eval_loss <= 2.0:
            return MoveQuality.MISTAKE
        else:
            return MoveQuality.BLUNDER
            
    def score_to_cp(self, score: chess.engine.PovScore, white_to_move: bool) -> float:
        """
        Convert engine score to centipawns from white's perspective
        
        Args:
            score: Engine evaluation score
            white_to_move: Whether it's white's turn
            
        Returns:
            Evaluation in centipawns from white's perspective
        """
        if score.is_mate():
            # Convert mate scores to large centipawn values
            mate_in = score.relative.mate()
            if mate_in is not None:
                if mate_in > 0:
                    # Positive mate means winning for the side to move
                    return 10000 - (mate_in * 10) if white_to_move else -10000 + (mate_in * 10)
                else:
                    # Negative mate means losing for the side to move
                    return -10000 - (mate_in * 10) if white_to_move else 10000 + (mate_in * 10)
        
        # Get centipawn value from the current player's perspective
        cp = score.relative.score()
        if cp is not None:
            # Convert to white's perspective
            return cp if white_to_move else -cp
        
        return 0
        
    def get_best_movelines(self, fen: str, num_moves: int = 5) -> Tuple[chess.Move, float, str]:
        """
        Get best move and evaluation for a position
        
        Args:
            fen: FEN string of the position
            num_moves: Number of moves to include in the line
            
        Returns:
            Tuple of (best_move, evaluation_in_cp, best_line_string)
        """
        # Set the FEN position
        self.board.set_fen(fen)
        
        # Analyze the position
        info = self.engine.analyse(self.board, chess.engine.Limit(depth=self.depth))
        
        # Get the score and best move
        score = info["score"]
        best_line_moves = info.get("pv", [])
        
        # Convert score to centipawns from white's perspective
        eval_cp = self.score_to_cp(score, self.board.turn == chess.WHITE)
        
        # Get best move
        best_move = best_line_moves[0] if best_line_moves else None
        
        # Convert best line to SAN
        san_line = []
        if best_line_moves:
            board_copy = self.board.copy()
            for move in best_line_moves[:num_moves]:
                san_line.append(board_copy.san(move))
                board_copy.push(move)
        
        return best_move, eval_cp, ' '.join(san_line)
    
    def get_line_after_move(self, fen: str, move: chess.Move, num_moves: int = 5) -> str:
        """
        Get the continuation line after a specific move
        
        Args:
            fen: Starting position FEN
            move: Move to make
            num_moves: Number of moves to analyze after
            
        Returns:
            String representation of the line
        """
        board = chess.Board(fen)
        move_san = board.san(move)
        board.push(move)
        
        # Get best continuation
        _, _, continuation = self.get_best_movelines(board.fen(), num_moves)
        
        if continuation:
            return f"{move_san} {continuation}"
        return move_san
        
    def analyze_game(self, game: chess.pgn.Game, player_name: str) -> Tuple[List[MoveAnalysis], List[DetailedMoveLog]]:
        """
        Analyze all moves of a specific player in a game with detailed logging
        
        Args:
            game: Chess game from PGN
            player_name: Name of player to analyze
            
        Returns:
            Tuple of (List of MoveAnalysis objects, List of DetailedMoveLog objects)
        """
        board = game.board()
        is_white = game.headers.get("White") == player_name
        mistakes = []
        detailed_logs = []
        
        move_count = 0
        
        print(f"\n{'='*80}")
        print(f"DETAILED MOVE-BY-MOVE ANALYSIS")
        print(f"{'='*80}")
        
        for node in game.mainline():
            move_count += 1
            full_move_number = (move_count + 1) // 2
            is_player_turn = (board.turn == chess.WHITE) == is_white
            
            # Store position FEN before the move
            position_fen = board.fen()
            
            # Get the actual move played
            actual_move = node.move
            move_played = board.san(actual_move)
            
            # Get the best move for this position
            best_move, eval_best_cp, best_line = self.get_best_movelines(position_fen)
            
            # Make the actual move and get its evaluation
            board.push(actual_move)
            after_actual_fen = board.fen()
            _, eval_after_actual_cp, actual_continuation = self.get_best_movelines(after_actual_fen)
            eval_after_actual_cp = -eval_after_actual_cp  # Flip since opponent's turn
            
            # Get line after actual move
            actual_move_full_line = f"{move_played} {actual_continuation}"
            
            # Initialize variables for best move evaluation
            eval_after_best_cp = eval_after_actual_cp
            eval_difference = 0
            
            # Only analyze if it's the player's move
            if is_player_turn:
                print(f"\nMove {full_move_number}. {'White' if board.turn == chess.BLACK else 'Black'}")
                print(f"  FEN: {position_fen}")
                print(f"  Actual move played: {move_played}")
                
                if best_move and best_move != actual_move:
                    # Evaluate position after best move
                    board.pop()  # Undo actual move
                    temp_board = chess.Board(position_fen)
                    best_move_san = temp_board.san(best_move)
                    board.push(best_move)
                    after_best_fen = board.fen()
                    _, eval_after_best_cp, _ = self.get_best_movelines(after_best_fen)
                    eval_after_best_cp = -eval_after_best_cp  # Flip since opponent's turn
                    board.pop()  # Undo best move
                    board.push(actual_move)  # Re-make actual move
                    
                    # Calculate evaluation difference
                    if is_white:
                        eval_difference = -(eval_after_best_cp - eval_after_actual_cp)
                    else:
                        eval_difference = -(eval_after_actual_cp - eval_after_best_cp)
                    
                    print(f"  Best move: {best_move_san}")
                    print(f"  Best move line: {best_line}")
                    print(f"  Eval after actual move: {eval_after_actual_cp/100:.2f}")
                    print(f"  Eval after best move: {eval_after_best_cp/100:.2f}")
                    print(f"  Actual move inferior line: {actual_move_full_line}")
                    print(f"  Evaluation difference: {eval_difference/100:.2f} pawns")
                    
                    # Convert to pawns for classification
                    eval_loss_pawns = eval_difference / 100.0
                    
                    # Store in detailed log
                    quality = self.classify_move_quality(eval_loss_pawns) if eval_loss_pawns > 0 else MoveQuality.BEST
                    
                    detailed_log = DetailedMoveLog(
                        move_number=full_move_number,
                        position_fen=position_fen,
                        actual_move_played=move_played,
                        best_move=best_move_san,
                        best_move_line=best_line,
                        eval_after_actual_move=eval_after_actual_cp / 100.0,
                        eval_after_best_move=eval_after_best_cp / 100.0,
                        actual_move_inferior_line=actual_move_full_line,
                        eval_difference=eval_loss_pawns,
                        is_player_move=True,
                        move_quality=quality.value
                    )
                    detailed_logs.append(detailed_log)
                    
                    # Add to mistakes if significant
                    if eval_loss_pawns > 0.3:  # Only record inaccuracies and worse
                        mistakes.append(MoveAnalysis(
                            move_number=full_move_number,
                            move_played=move_played,
                            best_move=best_move_san,
                            eval_before=eval_after_best_cp / 100.0,
                            eval_after=eval_after_actual_cp / 100.0,
                            eval_loss=eval_loss_pawns,
                            quality=quality,
                            best_line=best_line,
                            position_fen=position_fen,
                            actual_move_line=actual_move_full_line,
                            eval_after_best=eval_after_best_cp / 100.0
                        ))
                        
                        print(f"  >>> {quality.value.upper()} detected!")
                else:
                    # Best move was played
                    print(f"  Best move: {move_played} ✓")
                    print(f"  Best line: {best_line}")
                    print(f"  Evaluation: {eval_after_actual_cp/100:.2f}")
                    
                    detailed_log = DetailedMoveLog(
                        move_number=full_move_number,
                        position_fen=position_fen,
                        actual_move_played=move_played,
                        best_move=move_played,
                        best_move_line=best_line,
                        eval_after_actual_move=eval_after_actual_cp / 100.0,
                        eval_after_best_move=eval_after_actual_cp / 100.0,
                        actual_move_inferior_line=actual_move_full_line,
                        eval_difference=0,
                        is_player_move=True,
                        move_quality=MoveQuality.BEST.value
                    )
                    detailed_logs.append(detailed_log)
            else:
                # Opponent's move - just track it
                if self.enable_detailed_logging:
                    detailed_log = DetailedMoveLog(
                        move_number=full_move_number,
                        position_fen=position_fen,
                        actual_move_played=move_played,
                        best_move="",
                        best_move_line="",
                        eval_after_actual_move=eval_after_actual_cp / 100.0,
                        eval_after_best_move=0,
                        actual_move_inferior_line="",
                        eval_difference=0,
                        is_player_move=False,
                        move_quality=""
                    )
                    detailed_logs.append(detailed_log)
                
        return mistakes, detailed_logs

def save_detailed_analysis(all_games_data: List[Dict], output_file: str = "detailed_analysis.json"):
    """Save detailed analysis to JSON file"""
    with open(output_file, "w") as f:
        json.dump(all_games_data, f, indent=2)
    print(f"\nDetailed analysis saved to {output_file}")



In [11]:
with open('games.txt') as f:
    data = f.read()

In [5]:
!git clone https://github.com/official-stockfish/Stockfish.git

Cloning into 'Stockfish'...
remote: Enumerating objects: 39613, done.
remote: Counting objects: 100% (85/85), done.
remote: Compressing objects: 100% (28/28), done.
remote: Total 39613 (delta 64), reused 57 (delta 57), pack-reused 39528 (from 2)
Receiving objects: 100% (39613/39613), 18.33 MiB | 13.76 MiB/s, done.
Resolving deltas: 100% (30505/30505), done.


In [7]:
%cd Stockfish/src
!make -j profile-build

/Users/pgn001/Documents/chess/Data_prep_nbs/Stockfish/src
Successfully validated nn-1c0000000000.nnue
Successfully validated nn-37f18f62d772.nnue

Config:
debug: 'no'
sanitize: 'none'
optimize: 'yes'
arch: 'arm64'
bits: '64'
kernel: 'Darwin'
os: ''
prefetch: 'yes'
popcnt: 'yes'
pext: 'no'
sse: 'no'
mmx: 'no'
sse2: 'no'
ssse3: 'no'
sse41: 'no'
avx2: 'no'
avxvnni: 'no'
avx512: 'no'
vnni256: 'no'
vnni512: 'no'
avx512icl: 'no'
altivec: 'no'
vsx: 'no'
neon: 'yes'
dotprod: 'yes'
arm_version: '8'
lsx: 'no'
lasx: 'no'
target_windows: ''

Flags:
CXX: g++
CXXFLAGS:  -Wall -Wcast-qual -fno-exceptions -std=c++17  -pedantic -Wextra -Wshadow -Wmissing-declarations -m64 -mmacosx-version-min=10.15 -arch arm64 -DUSE_PTHREADS -DNDEBUG -O3 -funroll-loops -DIS_64BIT -DUSE_POPCNT -DUSE_NEON=8 -march=armv8.2-a+dotprod -DUSE_NEON_DOTPROD -DGIT_SHA=731ad9bb -DGIT_DATE=20250830 -DARCH=apple-silicon -flto=full
LDFLAGS:   -m64 -mmacosx-version-min=10.15 -arch arm64 -lpthread  -Wall -Wcast-qual -fno-exceptions -s

In [8]:
%cd ../..

/Users/pgn001/Documents/chess/Data_prep_nbs


In [12]:
data

'[Event "rated blitz game"]\n[Site "https://lichess.org/pHtTYXv7"]\n[Date "2025.08.29"]\n[White "usernam12"]\n[Black "Waldler59"]\n[Result "0-1"]\n[GameId "pHtTYXv7"]\n[UTCDate "2025.08.29"]\n[UTCTime "06:24:51"]\n[WhiteElo "2151"]\n[BlackElo "2275"]\n[WhiteRatingDiff "-4"]\n[BlackRatingDiff "+3"]\n[Variant "Standard"]\n[TimeControl "180+2"]\n[ECO "C25"]\n[Termination "Normal"]\n\n1. e4 d6 2. Nc3 e5 3. f4 exf4 4. Nf3 g5 5. Bc4 h6 6. O-O Bg7 7. d4 Ne7 8. Re1 O-O 9. h4 g4 10. Nh2 Nbc6 11. Bxf4 Bxd4+ 12. Kh1 Ng6 13. Bxh6 Qxh4 14. Ne2 Be5 0-1\n\n\n[Event "rated blitz game"]\n[Site "https://lichess.org/mTScJwdl"]\n[Date "2025.08.25"]\n[White "usernam12"]\n[Black "Ngcongo235"]\n[Result "1-0"]\n[GameId "mTScJwdl"]\n[UTCDate "2025.08.25"]\n[UTCTime "14:44:48"]\n[WhiteElo "2144"]\n[BlackElo "2156"]\n[WhiteRatingDiff "+7"]\n[BlackRatingDiff "-9"]\n[Variant "Standard"]\n[TimeControl "180+2"]\n[ECO "C29"]\n[Termination "Normal"]\n\n1. e4 e5 2. Nc3 Nf6 3. f4 d6 4. Nf3 Bg4 5. d4 exd4 6. Qxd4 Nc6 7. 

In [15]:
import chess.pgn
import io
import re

def parse_pgn_string(pgn_string):
    """
    Parse a string containing multiple PGN games and return a list of chess.pgn.Game objects.
    
    Args:
        pgn_string (str): String containing one or more PGN-formatted chess games
        
    Returns:
        list: List of chess.pgn.Game objects
    """
    games = []
    
    # Split the string into individual game strings
    # Games are typically separated by blank lines and start with [Event
    individual_games = split_pgn_into_games(pgn_string)
    
    print(f"Found {len(individual_games)} game(s) in the PGN string")
    
    for i, game_string in enumerate(individual_games, 1):
        try:
            # Create a StringIO object for this specific game
            game_io = io.StringIO(game_string)
            
            # Parse the game
            game = chess.pgn.read_game(game_io)
            
            if game is not None:
                games.append(game)
                white = game.headers.get('White', 'Unknown')
                black = game.headers.get('Black', 'Unknown')
                result = game.headers.get('Result', 'Unknown')
                print(f"Game {i}: {white} vs {black} - {result} ✓")
            else:
                print(f"Game {i}: Failed to parse (returned None)")
                
        except Exception as e:
            print(f"Game {i}: Error parsing - {str(e)}")
            # Try alternative parsing method
            try:
                game = parse_game_manually(game_string)
                if game:
                    games.append(game)
                    print(f"  -> Recovered using manual parsing")
            except:
                print(f"  -> Could not recover game")
    
    print(f"\nSuccessfully parsed {len(games)} out of {len(individual_games)} games")
    return games


def split_pgn_into_games(pgn_string):
    """
    Split a PGN string containing multiple games into individual game strings.
    
    Args:
        pgn_string (str): String containing multiple PGN games
        
    Returns:
        list: List of individual game strings
    """
    games = []
    current_game_lines = []
    
    lines = pgn_string.strip().split('\n')
    
    for line in lines:
        # Check if this line starts a new game (has [Event header)
        if line.strip().startswith('[Event ') and current_game_lines:
            # We've hit a new game, save the current one
            game_text = '\n'.join(current_game_lines).strip()
            if game_text:
                games.append(game_text)
            current_game_lines = [line]
        else:
            current_game_lines.append(line)
    
    # Don't forget the last game
    if current_game_lines:
        game_text = '\n'.join(current_game_lines).strip()
        if game_text:
            games.append(game_text)
    
    return games


def parse_game_manually(game_string):
    """
    Manually parse a PGN game string as a fallback method.
    
    Args:
        game_string (str): String containing a single PGN game
        
    Returns:
        chess.pgn.Game or None
    """
    try:
        # Extract headers
        headers = {}
        moves_text = ""
        lines = game_string.strip().split('\n')
        
        in_moves = False
        for line in lines:
            line = line.strip()
            if not line:
                continue
                
            if line.startswith('[') and line.endswith(']'):
                # Parse header
                match = re.match(r'\[(\w+)\s+"([^"]+)"\]', line)
                if match:
                    headers[match.group(1)] = match.group(2)
            else:
                # This is moves text
                in_moves = True
                moves_text += " " + line
        
        # Create a game object
        game = chess.pgn.Game()
        
        # Set headers
        for key, value in headers.items():
            game.headers[key] = value
        
        # Parse moves if we have them
        if moves_text.strip():
            # Clean the moves text
            moves_text = clean_moves_text(moves_text)
            
            # Try to parse the moves
            board = game.board()
            node = game
            
            # Split moves and process them
            tokens = moves_text.split()
            for token in tokens:
                # Skip move numbers and results
                if token in ['1-0', '0-1', '1/2-1/2', '*']:
                    break
                if '.' in token or token.isdigit():
                    continue
                    
                try:
                    # Try to parse as a move
                    move = board.parse_san(token)
                    node = node.add_variation(move)
                    board.push(move)
                except:
                    # Skip invalid moves
                    continue
        
        return game
        
    except Exception as e:
        print(f"Manual parsing failed: {e}")
        return None


def clean_moves_text(moves_text):
    """
    Clean moves text by removing comments and annotations.
    
    Args:
        moves_text (str): Raw moves text
        
    Returns:
        str: Cleaned moves text
    """
    # Remove comments in curly braces
    moves_text = re.sub(r'\{[^}]*\}', '', moves_text)
    
    # Remove variations in parentheses
    moves_text = re.sub(r'\([^)]*\)', '', moves_text)
    
    # Remove NAGs (like !?, +-, etc.)
    moves_text = re.sub(r'[!?]+', '', moves_text)
    moves_text = re.sub(r'\$\d+', '', moves_text)
    
    # Clean up whitespace
    moves_text = ' '.join(moves_text.split())
    
    return moves_text


# Alternative simplified version that's more forgiving
def parse_pgn_string_simple(pgn_string):
    """
    Simple PGN parser that's more forgiving of format issues.
    
    Args:
        pgn_string (str): String containing PGN games
        
    Returns:
        list: List of chess.pgn.Game objects
    """
    games = []
    
    # Try the standard approach first
    try:
        pgn_io = io.StringIO(pgn_string)
        while True:
            game = chess.pgn.read_game(pgn_io)
            if game is None:
                break
            games.append(game)
            
    except Exception as e:
        print(f"Standard parsing failed: {e}")
        print("Attempting recovery with individual game parsing...")
        
        # Fall back to parsing games individually
        individual_games = split_pgn_into_games(pgn_string)
        
        for game_string in individual_games:
            try:
                game_io = io.StringIO(game_string)
                game = chess.pgn.read_game(game_io)
                if game:
                    games.append(game)
            except:
                # Try manual parsing as last resort
                game = parse_game_manually(game_string)
                if game:
                    games.append(game)
    
    return games


# Test function to validate PGN format
def validate_pgn_format(pgn_string):
    """
    Validate and report issues with PGN format.
    
    Args:
        pgn_string (str): PGN string to validate
        
    Returns:
        dict: Validation report
    """
    report = {
        'total_lines': len(pgn_string.split('\n')),
        'games_found': 0,
        'issues': []
    }
    
    # Check for games
    games = split_pgn_into_games(pgn_string)
    report['games_found'] = len(games)
    
    # Check each game
    for i, game_string in enumerate(games, 1):
        lines = game_string.split('\n')
        
        # Check for headers
        headers_found = sum(1 for line in lines if line.strip().startswith('['))
        if headers_found == 0:
            report['issues'].append(f"Game {i}: No headers found")
        
        # Check for moves
        has_moves = any(not line.strip().startswith('[') and line.strip() 
                       for line in lines)
        if not has_moves:
            report['issues'].append(f"Game {i}: No moves found")
        
        # Check for result
        if '1-0' not in game_string and '0-1' not in game_string and '1/2-1/2' not in game_string:
            report['issues'].append(f"Game {i}: No result found")
    
    return report


# Usage example:
"""
# Read your PGN file
with open('games.txt', 'r') as f:
    pgn_string = f.read()

# Validate format first (optional)
validation = validate_pgn_format(pgn_string)
print("Validation Report:", validation)

# Parse the games
games = parse_pgn_string(pgn_string)

# If the main parser fails, try the simple version
if not games:
    print("Trying simplified parser...")
    games = parse_pgn_string_simple(pgn_string)

print(f"Successfully parsed {len(games)} games")

# Use with your analyzer
for game in games:
    print(f"Game: {game.headers.get('White')} vs {game.headers.get('Black')}")
"""

'\n# Read your PGN file\nwith open(\'games.txt\', \'r\') as f:\n    pgn_string = f.read()\n\n# Validate format first (optional)\nvalidation = validate_pgn_format(pgn_string)\nprint("Validation Report:", validation)\n\n# Parse the games\ngames = parse_pgn_string(pgn_string)\n\n# If the main parser fails, try the simple version\nif not games:\n    print("Trying simplified parser...")\n    games = parse_pgn_string_simple(pgn_string)\n\nprint(f"Successfully parsed {len(games)} games")\n\n# Use with your analyzer\nfor game in games:\n    print(f"Game: {game.headers.get(\'White\')} vs {game.headers.get(\'Black\')}")\n'

In [21]:

pgn_string = data
# Path to your Stockfish executable
STOCKFISH_PATH = 'Stockfish/src/stockfish'  # Update this to your path

# Parse games
games = parse_pgn_string(pgn_string)

# Analyze each game
player_name = "usernam12"
all_games_analysis = []

print(f"Analyzing games for {player_name}...\n")

with ChessAnalyzer(STOCKFISH_PATH, depth=15, enable_detailed_logging=True) as analyzer:
    for i, game in enumerate(games, 1):
        print(f"\n{'='*80}")
        print(f"GAME {i}: {game.headers.get('White')} vs {game.headers.get('Black')}")
        print(f"Result: {game.headers.get('Result')}")
        print(f"Playing as: {'White' if game.headers.get('White') == player_name else 'Black'}")
        print(f"{'='*80}")
        
        # Analyze the game
        mistakes, detailed_logs = analyzer.analyze_game(game, player_name)
        
        # Prepare comprehensive data
        game_analysis = {
            "game_info": {
                "game_number": i,
                "white": game.headers.get("White"),
                "black": game.headers.get("Black"),
                "result": game.headers.get("Result"),
                "player_color": "white" if game.headers.get("White") == player_name else "black",
                "player_won": (game.headers.get("Result") == "1-0" and game.headers.get("White") == player_name) or 
                             (game.headers.get("Result") == "0-1" and game.headers.get("Black") == player_name)
            },
            "summary": {
                "total_moves_analyzed": len([log for log in detailed_logs if log.is_player_move]),
                "perfect_moves": len([log for log in detailed_logs if log.is_player_move and log.move_quality == "best"]),
                "inaccuracies": len([m for m in mistakes if m.quality == MoveQuality.INACCURACY]),
                "mistakes": len([m for m in mistakes if m.quality == MoveQuality.MISTAKE]),
                "blunders": len([m for m in mistakes if m.quality == MoveQuality.BLUNDER])
            },
            "detailed_move_logs": [asdict(log) for log in detailed_logs if log.is_player_move],
            "mistakes_only": [
                {
                    "move_number": m.move_number,
                    "position_fen": m.position_fen,
                    "move_played": m.move_played,
                    "best_move": m.best_move,
                    "best_line": m.best_line,
                    "actual_move_line": m.actual_move_line,
                    "eval_after_actual": round(m.eval_after, 2),
                    "eval_after_best": round(m.eval_after_best, 2),
                    "evaluation_loss": round(m.eval_loss, 2),
                    "move_quality": m.quality.value
                }
                for m in mistakes
            ]
        }
        
        all_games_analysis.append(game_analysis)
        
        # Print summary
        print(f"\n{'='*80}")
        print(f"GAME SUMMARY")
        print(f"{'='*80}")
        print(f"Total moves analyzed: {game_analysis['summary']['total_moves_analyzed']}")
        print(f"Perfect moves: {game_analysis['summary']['perfect_moves']}")
        print(f"Inaccuracies: {game_analysis['summary']['inaccuracies']}")
        print(f"Mistakes: {game_analysis['summary']['mistakes']}")
        print(f"Blunders: {game_analysis['summary']['blunders']}")

# Save all analysis data
save_detailed_analysis(all_games_analysis, "usernam12_detailed_analysis.json")

# Also save a simplified version for quick LLM analysis
simplified_data = []
for game_data in all_games_analysis:
    simplified_game = {
        "game": game_data["game_info"],
        "summary": game_data["summary"],
        "key_mistakes": [
            {
                "move": m["move_number"],
                "fen": m["position_fen"],
                "played": m["move_played"],
                "best": m["best_move"],
                "loss": m["evaluation_loss"],
                "type": m["move_quality"]
            }
            for m in game_data["mistakes_only"]
            if m["evaluation_loss"] > 0.5  # Only significant mistakes
        ]
    }
    simplified_data.append(simplified_game)

with open("usernam12_simplified_for_llm.json", "w") as f:
    json.dump(simplified_data, f, indent=2)

print(f"\n{'='*80}")
print("ANALYSIS COMPLETE!")
print(f"{'='*80}")
print(f"Files created:")
print(f"  1. usernam12_detailed_analysis.json - Complete detailed analysis")
print(f"  2. usernam12_simplified_for_llm.json - Simplified version for LLM")
print(f"\nTotal games analyzed: {len(games)}")

# Suggest LLM prompt
print("\n" + "="*80)
print("SUGGESTED LLM PROMPT FOR PATTERN ANALYSIS:")
print("="*80)
print("""
Analyze the chess mistakes data and identify patterns in the player's errors. For each move, you have:
- FEN position before the move
- Actual move played vs best move
- Complete move lines showing consequences
- Evaluation differences

Please identify:
1. Tactical patterns missed (pins, forks, skewers, discovered attacks)
2. Positional weaknesses (pawn structure errors, piece placement issues)
3. Opening preparation gaps
4. Time-related patterns (do mistakes increase in time pressure?)
5. Specific piece configurations the player struggles with
6. Recurring strategic themes

Provide specific training recommendations based on these patterns.
""")


Found 50 game(s) in the PGN string
Game 1: usernam12 vs Waldler59 - 0-1 ✓
Game 2: usernam12 vs Ngcongo235 - 1-0 ✓
Game 3: usernam12 vs AK064 - 1-0 ✓
Game 4: usernam12 vs est8 - 1-0 ✓
Game 5: usernam12 vs IThror - 1-0 ✓
Game 6: usernam12 vs tekeltili - 1-0 ✓
Game 7: usernam12 vs study_hack - 0-1 ✓
Game 8: usernam12 vs eltvillekuhn - 1-0 ✓
Game 9: usernam12 vs chesshero01 - 0-1 ✓
Game 10: usernam12 vs KnownAsMage - 1-0 ✓
Game 11: usernam12 vs hapinok - 0-1 ✓
Game 12: usernam12 vs AndreiKanter - 0-1 ✓
Game 13: usernam12 vs pilliwarrior - 0-1 ✓
Game 14: usernam12 vs Agafonovtim - 0-1 ✓
Game 15: usernam12 vs nikolai-gertcyk - 0-1 ✓
Game 16: usernam12 vs LeoSuarezMDQ - 1-0 ✓
Game 17: usernam12 vs buzdolab - 1-0 ✓
Game 18: usernam12 vs faceface1 - 0-1 ✓
Game 19: usernam12 vs Makharu - 1-0 ✓
Game 20: usernam12 vs mitara - 0-1 ✓
Game 21: usernam12 vs azioneperpendicolare - 1-0 ✓
Game 22: usernam12 vs Vladimir63Iurev - 0-1 ✓
Game 23: usernam12 vs Miisuri - 1/2-1/2 ✓
Game 24: usernam12 vs dcp_35 